Utilizamos web scraping para automatizar las descargas en los PDFs de los datos estadisticos de la Universidad Nacional de Asunción

In [1]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import pandas as pd
from tqdm import tqdm

#### Creamos la Carpeta Automaticamente

In [3]:
RAW_DIR = Path('../data/raw/pdfs')

RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Carpeta Creada:', RAW_DIR)

Carpeta Creada: ..\data\raw\pdfs


#### Elegimos los PDFs de la pagina Estadistica UNA

In [8]:
BASE_URL = 'https://https://www.una.py/'
TARGET_PAGE = 'https://www.una.py/la-universidad/estadisticas/poblacion-carreras-de-grado-por-sedes'

#### Descargamos el HTML

In [9]:
response = requests.get(TARGET_PAGE)        #Envia solicitud HTTP GET p/ descargar contenido
print(response.status_code)                 # 200, servidor devuelve OK

200


#### Parseamos el HTML

In [10]:
soup = BeautifulSoup(response.text, 'html.parser')

#### Encontramos los enlaces PDF

In [11]:
pdf_links = []

for link in soup.find_all('a', href=True):
    href = link['href']
    if '.pdf' in href.lower():
        pdf_links.append(href)

len(pdf_links)

14

#### Normalización de URLs
En esta celda convierto URLs relativas a URLs absolutas

In [14]:
from urllib.parse import urljoin

full_links = []

for link in pdf_links:
    
    full_url = urljoin(BASE_URL, link)
    full_links.append(full_url)

full_links

['https://www.una.py/wp-content/uploads/2025/07/Postulantes-por-Unidad-Academica-2024.pdf',
 'https://www.una.py/wp-content/uploads/2025/07/Postulantes-por-Unidad-Academica-Carreras-y-Sedes-2024.pdf',
 'https://www.una.py/wp-content/uploads/2025/07/Postulantes-por-Carreras-y-Sedes-2020-2024.pdf',
 'https://www.una.py/wp-content/uploads/2025/07/Ingresantes-por-Unidad-Academica-2020-2024.pdf',
 'https://www.una.py/wp-content/uploads/2025/07/Ingresantes-por-Unidad-Academica-Carreras-y-Sedes-2020-2024.pdf',
 'https://www.una.py/wp-content/uploads/2025/07/Ingresantes-por-Carreras-y-Sedes-2020-2024.pdf',
 'https://www.una.py/wp-content/uploads/2025/07/Matriculados-por-Unidad-Academica-2020-2024.pdf',
 'https://www.una.py/wp-content/uploads/2025/07/Matriculados-por-Unidad-Academica-Carreras-y-Sedes-2020-2024.pdf',
 'https://www.una.py/wp-content/uploads/2025/07/Matriculados-por-Carreras-y-Sedes-2020-2024.pdf',
 'https://www.una.py/wp-content/uploads/2025/07/Egresados-por-Unidad-Academica-2020

#### Descargamos PDFs
Lo hacemos de forma masiva desde una lista y registrar el exito o falla de cada una

In [16]:
metadata = []                           # guarda infomacion: nombre, url, status

for url in tqdm(full_links):            #tqdm agrega barras de progreso

    try:
        filename = url.split('/')[-1]
            
        filepath = RAW_DIR / filename

        if filepath.exists():               # Control de existencia
            print(f"{filename} ya existe")
            metadata.append({
                "filename": filename,
                "url": url,
                "status": "ya existe" #
            })
            continue

        response = requests.get(url)            # descargamos el contenido del url

        with open(filepath, 'wb') as f:
            f.write(response.content)           # crea el archivo en modo escritura binaria

        metadata.append({                       # registrar éxito
            'filename': filename,
            'url'     : url,
            'status'  : 'downloaded'
        })

    except Exception as e:                      # registrar fracaso
        metadata.append({
            'filename' : filename,
            'url'      : url,
            'status'   : str(e)
        })
        


100%|██████████| 14/14 [00:00<00:00, 3622.47it/s]

Postulantes-por-Unidad-Academica-2024.pdf ya existe
Postulantes-por-Unidad-Academica-Carreras-y-Sedes-2024.pdf ya existe
Postulantes-por-Carreras-y-Sedes-2020-2024.pdf ya existe
Ingresantes-por-Unidad-Academica-2020-2024.pdf ya existe
Ingresantes-por-Unidad-Academica-Carreras-y-Sedes-2020-2024.pdf ya existe
Ingresantes-por-Carreras-y-Sedes-2020-2024.pdf ya existe
Matriculados-por-Unidad-Academica-2020-2024.pdf ya existe
Matriculados-por-Unidad-Academica-Carreras-y-Sedes-2020-2024.pdf ya existe
Matriculados-por-Carreras-y-Sedes-2020-2024.pdf ya existe
Egresados-por-Unidad-Academica-2020-2024.pdf ya existe
Egresados-por-Unidad-Academica-Carreras-y-Sedes-2020-2024.pdf ya existe
Egresados-por-Carreras-y-Sedes-2020-2024.pdf ya existe
Resolucion-N°-3448-Calendario-de-feriados-y-asuetos-UNA.pdf ya existe
Resolucion-N°-3448-Calendario-de-feriados-y-asuetos-UNA.pdf ya existe


#### Guardamos el metadata
Mediante panda los convertimos a dataframe y luego a formato .csv

In [17]:
df_meta = pd.DataFrame(metadata)

df_meta.to_csv(
    '../data/raw/download_log.csv',     #registro de descarga
    index=False
)

#### Vemos las primeras filas del CSV con el metadata

In [22]:
df_meta.head()

,filename,url,status
0,Postulantes-por-Unidad-Academica-2024.pdf,https://www.una.py/wp-content/uploads/2025/07/...,ya existe
1,Postulantes-por-Unidad-Academica-Carreras-y-Se...,https://www.una.py/wp-content/uploads/2025/07/...,ya existe
2,Postulantes-por-Carreras-y-Sedes-2020-2024.pdf,https://www.una.py/wp-content/uploads/2025/07/...,ya existe
3,Ingresantes-por-Unidad-Academica-2020-2024.pdf,https://www.una.py/wp-content/uploads/2025/07/...,ya existe
4,Ingresantes-por-Unidad-Academica-Carreras-y-Se...,https://www.una.py/wp-content/uploads/2025/07/...,ya existe
